# 02 — Dataset Builder

Converts raw collected text into three training-ready datasets:

1. **SFT corpus** — `data/processed/sft_rf_domain.jsonl`  
   Chat-format pairs: `{instruction, input, output}` covering RF design Q&A,  
   circuit sizing, layout code generation, antenna design, link budgets.

2. **Continued pre-training corpus** — `data/processed/pretrain_rf.jsonl`  
   Raw RF text (books, papers, code) chunked for next-token prediction.

3. **PINN dataset** — `data/processed/pinn_antenna.csv`  
   Geometry parameters → S11/gain/efficiency for training the EM surrogate model.

In [ ]:
!pip install tqdm pandas scikit-learn requests

In [2]:
import json, re, random, os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

random.seed(42)

RAW_DIR = Path('../data/raw')
PROC_DIR = Path('../data/processed')
SYNTH_DIR = Path('../data/synthetic')
for d in [PROC_DIR, SYNTH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Continued Pre-Training Corpus — Chunk Raw Text

In [ ]:
CHUNK_SIZE = 2048   # tokens ~ chars/4
CHUNK_CHARS = CHUNK_SIZE * 4
OVERLAP = 256 * 4

def chunk_text(text: str, chunk_size: int = CHUNK_CHARS, overlap: int = OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if len(chunk) > 200:  # skip tiny fragments
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

pretrain_records = []

# Books + arXiv text
for txt_file in list((RAW_DIR / 'texts').glob('*.txt')) + list((RAW_DIR / 'arxiv' / 'texts').glob('*.txt')):
    text = txt_file.read_text(errors='ignore')
    source = txt_file.stem
    for chunk in chunk_text(text):
        pretrain_records.append({'text': chunk, 'source': source})

# arXiv abstracts
abstracts_file = RAW_DIR / 'arxiv' / 'all_abstracts.txt'
if abstracts_file.exists():
    text = abstracts_file.read_text(errors='ignore')
    for chunk in chunk_text(text, chunk_size=1024*4, overlap=128*4):
        pretrain_records.append({'text': chunk, 'source': 'arxiv_abstracts'})

# RF + GLayout code corpus — scan actual source code repos
CODE_ROOTS = [
    RAW_DIR / 'code' / 'analogcoder' / 'sample_design',
    RAW_DIR / 'code' / 'analogcoder' / 'subcircuit_lib',
    RAW_DIR / 'code' / 'analogcoder' / 'problem_check',
    RAW_DIR / 'code' / 'chipster' / 'data' / 'analog_datasets',
    RAW_DIR / 'code' / 'glayout_llm' / 'src',
    RAW_DIR / 'code' / 'glayout_ihp130',
    RAW_DIR / 'code' / 'openfasoc',
    RAW_DIR / 'code' / 'skrf',
]
code_files_loaded = 0
for root in CODE_ROOTS:
    if not root.exists():
        continue
    for py_file in root.rglob('*.py'):
        try:
            text = py_file.read_text(errors='ignore')
            if len(text) < 100:
                continue
            source_tag = py_file.parent.parent.name + '/' + py_file.name
            for chunk in chunk_text(text, chunk_size=4096, overlap=512):
                pretrain_records.append({'text': chunk, 'source': source_tag})
            code_files_loaded += 1
        except Exception:
            pass
print(f'Code corpus: {code_files_loaded} Python files scanned')

random.shuffle(pretrain_records)
out = PROC_DIR / 'pretrain_rf.jsonl'
with open(out, 'w') as f:
    for r in pretrain_records:
        f.write(json.dumps(r) + '\n')

print(f'Pre-training corpus: {len(pretrain_records):,} chunks -> {out}')

## 2. SFT Dataset — Handcrafted RF Design Q&A Templates

These are expert-written instruction-response pairs covering the core RF design tasks.

In [ ]:
# === HANDCRAFTED RF EXPERT Q&A ===
# Each entry: {instruction, input (optional context), output}

HANDCRAFTED_QA = [
    # --- LNA Design ---
    {
        "instruction": "Design a Ka-band (28 GHz) cascode LNA in 65nm CMOS targeting NF < 3 dB and gain > 20 dB. Specify topology, transistor sizing, matching network approach, and bias conditions.",
        "input": "",
        "output": """Ka-band cascode LNA design in 65nm CMOS (28 GHz):

**Topology:** Inductively-degenerated common-source (CS) input stage + common-gate (CG) cascode to maximize gain-bandwidth and improve reverse isolation.

**Transistor sizing:**
- M1 (CS): W = 40 µm, L = 60 nm, Nf = 20 fingers (W/f = 2 µm)
  - At 28 GHz, optimal Wf ≈ 1/(2π·f·Rg·Cgs) → 2 µm fingers minimize Rg contribution to NF
- M2 (CG): W = 40 µm, L = 60 nm, same sizing to maintain current drive

**Bias:** Vdd = 1.2 V, IDS ≈ 10 mA (IDS/Wg ≈ 0.25 mA/µm), Vgs − Vth ≈ 0.15 V (near peak-fT region)

**Matching network:**
- Input: Ls (source degeneration, 80 pH) sets real part of Zin = 50 Ω; Lg (gate inductor, 0.3 nH) resonates out imaginary part at 28 GHz
- Inter-stage: transmission-line transformer or shunt inductor for impedance transformation between CS and CG
- Output: shunt L + series C matching to 50 Ω load

**Expected performance:**
- NF: 2.5–3.0 dB (dominated by M1 drain current noise: NF_min ≈ 1 + 2·γ·ω/ωT·√(gm·Rn))
- Gain: 20–22 dB
- S11 < −10 dB
- IIP3 ≈ −5 dBm (limited by M1 nonlinearity)
- PDC ≈ 12 mW

**Layout considerations:**
- Use multi-finger transistors with gate contacted on both sides to halve Rg
- Route inductors in top-metal (thick Cu) with minimum substrate coupling
- Add substrate contacts (deep N-well isolation if available) around LNA core
- Use differential shielded inductors to reject common-mode substrate noise"""
    },
    # --- Phased Array ---
    {
        "instruction": "Explain the phase shifter architectures used in Ka-band CMOS phased arrays for satellite receivers, with tradeoffs.",
        "input": "",
        "output": """Ka-band CMOS phase shifter architectures for satellite phased arrays:

**1. Switched delay lines (true-time delay, TTD)**
- Operation: Select between transmission line segments of different lengths via RF switches (CMOS or FET)
- Advantage: Frequency-independent phase shift (no squint) — critical for wideband satellite signals
- Disadvantage: Area scales with bandwidth and frequency; insertion loss ~3–8 dB at Ka-band

**2. LC all-pass networks (vector modulator)**
- Operation: 90° hybrid splits signal into I/Q; variable attenuators on each path; recombine
- Advantage: Continuous phase resolution, amplitude control simultaneously
- Disadvantage: NF degradation, complex calibration, ~6–10 dB insertion loss

**3. Reflective-type phase shifter (RTPS)**
- Operation: 90° hybrid + variable reflection loads (varactors or switched caps)
- Advantage: Low IL (~2–4 dB), compact, good linearity
- Disadvantage: Limited phase range (~180°), amplitude-phase coupling

**4. Magnetic-tuning phase shifter (used in ISSCC 2022 Ka-band chip)**
- Operation: Current-controlled ferromagnetic material (permalloy) integrated on-chip shifts inductor permeability → shifts resonance frequency → shifts phase
- Advantage: No active devices in signal path → very low NF impact; radiation-hard (no carrier trapping)
- Disadvantage: Requires special process integration; limited tuning range

**5. Active phase rotator (digital)**
- Operation: Digitally weight I/Q components using current-mode DACs before combination
- Advantage: Digital control, easy calibration, fine resolution (6–8 bit)
- Disadvantage: Highest power, NF ≈ LNA NF + 3–6 dB degradation

**Recommendation for LEO satellite receiver (Ka-band, 26–30 GHz):**
RTPS or switched capacitor networks for best NF/area/power tradeoff at 65nm node.
Magnetic-tuning if radiation hardness is the primary constraint."""
    },
    # --- Antenna ---
    {
        "instruction": "Design a microstrip patch antenna for Ka-band at 28 GHz on Rogers RO4003C substrate (εr=3.55, h=0.508mm). Give all dimensions.",
        "input": "",
        "output": """Microstrip patch antenna design for 28 GHz (Ka-band) on RO4003C:

**Substrate:** εr = 3.55, tanδ = 0.0027, h = 0.508 mm

**Step 1 — Patch width W:**
W = (c / 2f) × √(2 / (εr + 1))
W = (3×10⁸ / 2×28×10⁹) × √(2 / 4.55) = 5.357 mm × 0.663 = **3.55 mm**

**Step 2 — Effective permittivity εeff:**
εeff = (εr+1)/2 + (εr−1)/2 × (1 + 12h/W)^(-0.5)
εeff = 2.275 + 1.275 × (1 + 12×0.508/3.55)^(-0.5) = 2.275 + 1.275×0.611 = **3.054**

**Step 3 — Effective patch length Leff:**
Leff = c / (2f × √εeff) = 3×10⁸ / (56×10⁹ × 1.748) = **3.064 mm**

**Step 4 — Length extension ΔL:**
ΔL = 0.412h × (εeff+0.3)(W/h+0.264) / ((εeff−0.258)(W/h+0.8))
ΔL = 0.412×0.508 × (3.354×7.137) / (2.796×7.813) = 0.209 × 23.94/21.84 = **0.229 mm**

**Step 5 — Physical patch length L:**
L = Leff − 2ΔL = 3.064 − 0.458 = **2.61 mm**

**Final dimensions:** W = 3.55 mm, L = 2.61 mm

**Feed:** Inset microstrip feed
- Feed line width (50 Ω): w₀ = 1.16 mm (for RO4003C h=0.508mm)
- Inset depth y₀ = L/π × arccos(√(Rin/50)) where Rin ≈ 200 Ω → y₀ ≈ 0.87 mm
- Inset slot width: 0.2 mm

**Expected performance:**
- Resonant freq: 28 GHz ±2%
- Gain: ~8 dBi
- BW (−10 dB S11): ~800 MHz (2.9%)
- Radiation efficiency: ~85–90% (limited by tanδ loss)"""
    },
    # --- GLayout code generation ---
    {
        "instruction": "Write GLayout Python code to generate a simple RF bypass capacitor cell in GF180MCU PDK using gdsfactory.",
        "input": "",
        "output": """```python
import gdsfactory as gf
from glayout.flow.pdk.mappedpdk import MappedPDK
from glayout.flow.primitives.mimcap import mimcap


@gf.cell
def rf_bypass_cap(
    pdk: MappedPDK,
    width: float = 20.0,   # um
    height: float = 20.0,  # um
    fingers: int = 4,
) -> gf.Component:
    \"\"\"RF bypass capacitor using MIM cap array.
    
    Target: ~1 pF at Ka-band for supply bypass / matching.
    MIM cap density in GF180: ~1 fF/µm², so 20×20 µm = 400 fF per finger.
    4 fingers in parallel → ~1.6 pF.
    \"\"\"
    c = gf.Component()
    
    finger_width = width
    finger_height = height / fingers
    spacing = pdk.get_grule('capm')['min_separation'] if 'capm' in pdk.grules else 2.0
    
    ports_top = []
    ports_bot = []
    
    for i in range(fingers):
        cap = c << mimcap(pdk, width=finger_width, height=finger_height)
        cap.movey(i * (finger_height + spacing))
        ports_top.append(cap.ports['top_met_N'])
        ports_bot.append(cap.ports['bottom_met_S'])
    
    # Connect all top plates (RF signal) together
    c.add_port('RF', port=ports_top[0])
    # Connect all bottom plates (GND) together  
    c.add_port('GND', port=ports_bot[0])
    
    c.flatten()
    return c


if __name__ == '__main__':
    from glayout.flow.pdk.gf180_mapped import gf180
    cap = rf_bypass_cap(pdk=gf180, width=20, height=20, fingers=4)
    cap.show()  # Opens in KLayout
    print(f'Bypass cap area: {cap.area():.1f} µm²')
```"""
    },
    # --- Link budget ---
    {
        "instruction": "Calculate the link budget for a Ka-band (28 GHz) LEO satellite downlink at 550 km altitude. EIRP = 40 dBW, antenna gain = 35 dBi, data rate = 100 Mbps. Is the link closed?",
        "input": "",
        "output": """Ka-band LEO downlink link budget (28 GHz, 550 km altitude):

**Transmit side (satellite):**
- EIRP: 40 dBW

**Free-space path loss (FSPL):**
FSPL = 20·log₁₀(4πd/λ) = 20·log₁₀(4π × 550×10³ / (3×10⁸/28×10⁹))
λ = 10.7 mm → FSPL = 20·log₁₀(4π × 550000 / 0.0107) = 20·log₁₀(6.45×10⁸) = **176.2 dB**

**Atmospheric losses:**
- Rain fade (Ka-band, moderate rain zone, 99.9% availability): 3 dB
- Gaseous absorption (O₂ + H₂O at 28 GHz): 0.5 dB
- Total atmospheric: **3.5 dB**

**Receive side (ground terminal):**
- Rx antenna gain: 35 dBi (phased array, ~50 cm aperture)
- Rx system noise temperature: 290 K → Tsys = 290 K → N₀ = kTsys = −174 + 10·log(290/290) + NF
- Rx NF: 3 dB → Tsys_total ≈ 580 K → N₀ = −174 + 10·log(2) = **−171 dBW/Hz**
- G/T = 35 − 10·log(580) = 35 − 27.6 = **7.4 dB/K**

**Received C/N₀:**
C/N₀ = EIRP − FSPL − Latm + G/T − k
= 40 − 176.2 − 3.5 + 35 − 3 − (−228.6)
= **121.9 dBHz**

**Required C/N₀ for 100 Mbps:**
Using DVB-S2 with 8PSK, code rate 2/3 → Eb/N₀ required = 6.5 dB
C/N₀_required = Eb/N₀ + 10·log(Rb) = 6.5 + 80 = **86.5 dBHz**

**Margin:** 121.9 − 86.5 = **+35.4 dB** ✓ Link is CLOSED with large margin.

This margin allows for:
- More aggressive modulation (32APSK, 64APSK) to increase spectral efficiency
- Smaller/cheaper ground terminal antenna
- Reduced transmit power to extend satellite battery life"""
    },
    # --- Radiation hardening ---
    {
        "instruction": "What techniques are used to make CMOS RF ICs radiation-hardened for LEO satellite applications?",
        "input": "",
        "output": """Radiation hardening techniques for CMOS RF ICs in LEO (total dose ~10–100 krad/year):

**1. Layout-level hardening (RHBD — Radiation Hardened By Design)**
- **Enclosed gate transistors (ELT):** Transistor gate surrounds the drain completely, eliminating parasitic edge leakage paths caused by STI (shallow trench isolation) oxide charge buildup. Eliminates radiation-induced leakage current increase.
- **Guard rings:** P+ rings around NMOS and N+ rings around PMOS to collect minority carriers from heavy ion strikes, preventing latch-up.
- **Single-event upset (SEU) mitigation:** Triple modular redundancy (TMR) on digital control registers; interleaved flip-flop layout to spread critical nodes.

**2. Bias point selection**
- Radiation shifts Vth of NMOS negative (oxide charge) and PMOS positive. Design bias circuits to tolerate ±100 mV Vth shift without performance collapse.
- Use self-biased topologies with PTAT current references that track Vth shifts.

**3. Magnetic-tuning components**
- Avoid varactors (MOS capacitors are sensitive to oxide charge) in phase shifters — substitute with magnetically-tuned inductors (ISSCC 2022 technique).
- Use switched-capacitor arrays with metal-insulator-metal (MIM) caps instead of MOS caps.

**4. Process selection**
- Thin gate oxide processes (65 nm, 28 nm) are inherently more radiation tolerant (fewer oxide traps) than older thick-oxide processes.
- SOI processes offer improved latch-up immunity but introduce floating body effects.

**5. Shielding**
- On-chip: Deep N-well isolation under sensitive analog blocks
- Module-level: Tantalum or aluminum shielding attenuates proton/electron flux

**Key spec:** Target TID tolerance > 100 krad(Si), SEL immunity > 60 MeV·cm²/mg LET threshold."""
    },
    # --- Phased array combiner ---
    {
        "instruction": "Design an 8:1 RF signal combiner for a Ka-band phased array receiver using a binary Wilkinson tree. How many stages are needed and what is the total insertion loss?",
        "input": "",
        "output": """8:1 RF Combiner using binary Wilkinson tree (Ka-band, 28 GHz):

**Structure:** 3 stages of 2:1 Wilkinson dividers/combiners connected in a binary tree.
- Stage 1: 4× 2:1 combiners (8 inputs → 4 outputs)
- Stage 2: 2× 2:1 combiners (4 → 2)
- Stage 3: 1× 2:1 combiner (2 → 1)
Total: **7 Wilkinson cells** for 8:1 combining.

**2:1 Wilkinson combiner at 28 GHz:**
- Quarter-wave transformer: λ/4 at 28 GHz = (3×10⁸)/(4×28×10⁹) = 2.68 mm in air
  In microstrip on GF180 (εeff ≈ 3.5): λ/4 ≈ 1.43 mm
- Transformer impedance: Z0·√2 = 50·1.414 = 70.7 Ω
- Isolation resistor: R = 2·Z0 = 100 Ω (polysilicon on-chip)
- Insertion loss per stage: ~0.3 dB (3.5 dB theoretical + mismatch + metal loss)

**Total 8:1 combiner insertion loss:**
- Ideal combining loss: 10·log(8) = 9.03 dB (8→1 power split)
- Metal/dielectric loss per stage: ~0.5 dB × 3 stages = 1.5 dB
- Total insertion loss from single input to output: **~1.5 dB** (excluding the inherent 9 dB combining ratio)

**Coherent combining gain:**
- 8 in-phase signals combine: power = 8×P_per_element → 9 dB gain over single element
- Net received SNR improvement over single element: 9 dB − 1.5 dB = **+7.5 dB**

**Layout in GF180:**
- Use metal5/metal6 (thickest) for transmission lines to minimize resistive loss
- Keep all 8 input paths equal length (matched delay) for phase coherence
- Total area estimate: ~(1.5 mm × 3) × tree width ≈ **0.8 mm² for complete 8:1 tree**"""
    },
    # --- MTPS ---
    {
        "instruction": "Design a 5-bit switched-capacitor phase shifter (MTPS) in GF180MCU for Ka-band (28 GHz) phased array. Specify capacitor values, switch sizing, and phase resolution.",
        "input": "",
        "output": """5-bit Switched-Capacitor Phase Shifter for 28 GHz in GF180MCU:

**Principle:** Binary-weighted MIM capacitor array switched by NMOS transistors.
Each bit k provides phase shift: Δφ_k = 2^(k-1) × LSB

**Phase range and resolution:**
- Total range: 360° / 8 elements ≈ 45° per element → need 180° range (with ±90° steering)
- Actual: 0–337.5° (5-bit), LSB = 360°/32 = **11.25°**

**Capacitor sizing (reflective-type, high-Q tank):**
Phase shift: Δφ = 2·arctan(ΔX/50) where ΔX is susceptance change per bit
LSB capacitance: CLSB = tan(LSB/2) / (2π·f·50) = tan(5.625°) / (2π×28×10⁹×50)
= 0.0985 / (8.796×10⁹) = **11.2 fF**

Capacitor array (MIM in GF180, ~1 fF/µm²):
- Bit 0 (LSB,  11.25°): C0 =  11.2 fF → 11.2 µm²
- Bit 1 (22.5°):        C1 =  22.4 fF → 22.4 µm²
- Bit 2 (45°):          C2 =  44.8 fF → 44.8 µm²
- Bit 3 (90°):          C3 =  89.6 fF → 89.6 µm²
- Bit 4 (MSB, 180°):    C4 = 179.2 fF → 179.2 µm²

**NMOS switch sizing (GF180, L=0.18µm):**
Switch Ron target: < 5 Ω (to maintain Q > 10 at 28 GHz)
Ron ≈ 1/(µn·Cox·W/L·(Vgs−Vth)); target W ≥ 20 µm for MSB, 5 µm for LSB
- Bit 4 switch: W = 40 µm, Fingers = 20 (Ron ≈ 2 Ω)
- Bit 0 switch: W =  8 µm, Fingers =  4 (Ron ≈ 10 Ω, acceptable for LSB)

**Insertion loss budget:**
- Switch Ron loss at 28 GHz: ~1.5–2.5 dB total
- MIM cap Q at 28 GHz (GF180): Q ≈ 30 → additional 0.5 dB loss
- Total MTPS insertion loss: **2–3 dB** (add LNA NF accordingly)

**Control:** 5-bit digital thermometer or binary code from baseband DSP via level shifter (1.8V CMOS → 3.3V switch drive for lower Ron)"""
    },
    # --- RX chain ---
    {
        "instruction": "Describe the complete Ka-band satellite receiver front-end signal chain from antenna to ADC input, with gain/NF at each stage. Target system NF < 4 dB.",
        "input": "",
        "output": """Ka-band satellite RX front-end signal chain (28 GHz, GF180 CMOS):

**Chain:** Antenna → LNA → Buffer → Phase Shifter → Combiner → RF Amp → Mixer → IF Filter → ADC

| Stage | Gain (dB) | NF (dB) | P1dB (dBm) | Notes |
|-------|-----------|---------|------------|-------|
| Antenna | 10 | — | — | Patch array element, 10 dBi |
| LNA | +18 | 2.5 | −5 | Cascode, first noise-critical stage |
| Buffer | +0 | 3.5 | +0 | Source follower, drives 50Ω PS |
| Phase shifter (MTPS) | −2.5 | 2.5 | +5 | Switched-cap, 5-bit |
| 8:1 Combiner | −9+7.5 = −1.5 | — | — | Coherent combining gain |
| RF Amplifier | +15 | 4.0 | +5 | After combiner; noise less critical |
| Downconvert mixer | −6 | 8.0 | +5 | Gilbert cell, image-reject |
| IF filter (50 MHz BW) | −3 | 3.0 | — | Active Gm-C or passive LC |

**Cascaded NF (Friis formula):**
NF_sys = NF1 + (NF2−1)/G1 + (NF3−1)/(G1·G2) + ...
= 2.5 + (3.5−1)/63 + (2.5−1)/(63×0.56) + ...
= 2.5 + 0.040 + 0.042 + ... ≈ **2.6 dB** ✓ (< 4 dB target)

**Key design insight:**
- LNA noise figure dominates; everything after is attenuated by LNA gain.
- Buffer must have very low noise (added NF ÷ LNA_gain) and good IIP3 to avoid compressing before MTPS.
- Phase shifter's 2.5 dB loss adds only (2.5−1)/316 = 0.005 dB to system NF.
- After 8:1 combiner, the system can afford higher-NF amplifiers.

**ADC requirements:**
- Signal BW: 500 MHz (DVB-S2X channel)
- ENOB: 8 bits sufficient (SNR ≈ 50 dB)
- Sample rate: 1.2 Gsps
- IF frequency: 1–2 GHz (choose for on-chip LC filter practicality)"""
    },
    # --- GLayout phased array cell ---
    {
        "instruction": "Write GLayout Python code for an RF buffer/source-follower cell in GF180MCU using nmos primitives.",
        "input": "",
        "output": """```python
import gdsfactory as gf
from gelochip.glayout.pdk.mappedpdk import MappedPDK
from gelochip.glayout.primitives.fet import nmos
from gelochip.glayout.primitives.mimcap import mimcap
from gelochip.glayout.util.comp_utils import prec_ref_center, movex, movey
from gelochip.glayout.util.port_utils import rename_ports_by_orientation
from gelochip.glayout.spice.netlist import Netlist


@gf.cell
def rf_buffer(
    pdk: MappedPDK,
    width: float = 20.0,
    fingers: int = 8,
    ac_cap_size: tuple = (10.0, 10.0),
    sd_rmult: int = 2,
) -> gf.Component:
    \"\"\"NMOS source follower RF buffer (unity gain, low output impedance).
    
    Topology: Vin → [AC-cap] → Gate → NMOS → Source → Vout
    Biased with a current source NMOS at source node.
    
    Target (GF180, 5GHz):
        Gain: ~−1 dB (source follower, Av ≈ gm·Rout/(1+gm·Rout))
        NF: ~3.5 dB
        IIP3: +5 dBm
    \"\"\"
    top = gf.Component()
    
    # Main source-follower transistor
    m_sf = nmos(pdk, width=width, fingers=fingers, sd_rmult=sd_rmult)
    # Tail current source
    m_tail = nmos(pdk, width=width/2, fingers=fingers//2, sd_rmult=sd_rmult)
    # AC coupling capacitor at input
    c_in = mimcap(pdk, size=ac_cap_size)
    
    r_sf   = prec_ref_center(m_sf)
    r_tail = prec_ref_center(m_tail)
    r_cin  = prec_ref_center(c_in)
    
    top.add(r_sf)
    top.add(r_tail)
    top.add(r_cin)
    
    bbox_h = m_sf.bbox[1][1] - m_sf.bbox[0][1]
    bbox_w = m_sf.bbox[1][0] - m_sf.bbox[0][0]
    sep = pdk.get_grule('met2')['min_separation']
    
    movey(r_tail, -(bbox_h + sep))
    movex(r_cin, -(bbox_w + sep))
    
    top.add_ports(r_sf.get_ports_list(),   prefix='sf_')
    top.add_ports(r_tail.get_ports_list(), prefix='tail_')
    top.add_ports(r_cin.get_ports_list(),  prefix='cin_')
    
    top.info['specs'] = {
        'topology': 'source_follower_buffer',
        'target_gain_dB': -1.0,
        'target_nf_dB': 3.5,
        'target_iip3_dBm': 5.0,
    }
    top.info['netlist'] = Netlist(
        circuit_name='RF_BUFFER',
        nodes=['RFIN', 'RFOUT', 'VBIAS', 'VDD', 'VSS'],
    )
    return rename_ports_by_orientation(top)


if __name__ == '__main__':
    from gelochip.glayout.pdk.gf180_mapped import gf180_mapped_pdk as pdk
    buf = rf_buffer(pdk, width=20.0, fingers=8)
    buf.show()
```"""
    },
    # --- Satellite system architecture ---
    {
        "instruction": "What is the architecture of the Ka-band phased array transceiver chip shown in the ISSCC paper for a small satellite? Describe the 8-element receive path in detail.",
        "input": "Chip: 8-element/chipx32 → 256-array phased array CMOS chip for Ka-band LEO satellite, 1.5mm × 4mm die.",
        "output": """Ka-band 8-element phased array receiver chip architecture (per chip):

**System overview:**
- 8 receive (RX) elements per chip × 32 chips = 256-element phased array
- Die size: 4mm × 1.5mm (6 mm² in 65nm or 28nm CMOS)
- Operating frequency: Ka-band (26.5–40 GHz, likely 28 GHz downlink)

**Per-element RX chain (×8):**
1. **Multi-Coupling LNA (Multi-Coup. LNA)**
   - LNA with multi-coupled inductors for broadband impedance match
   - NF ≈ 2.5 dB, Gain = 18 dB, uses transformer feedback for noise/gain optimization
   - Input matched to patch antenna (50 Ω), output into phase shifter

2. **Buffer + Phase Shifter (BUF+PS)**
   - Source follower buffer: Gain ≈ 0 dB, drives 50Ω phase shifter input
   - Phase shifter: analog vector modulator (I/Q weighting) or switched-cap
   - Phase resolution: 6-bit (5.625° LSB), 0–360° range
   - Combined block NF ≈ 4 dB, insertion loss ≈ 3 dB

3. **MTPS (Multi-Throw Phase Shifter)**
   - Separate high-linearity phase trimmer for fine calibration
   - Used to compensate element-to-element process variation
   - Digitally controlled, stored in on-chip MTP (Multi-Time Programmable) registers
   
**Per-chip combining path:**
4. **8:1 Σ Combiner** (center of chip)
   - Binary Wilkinson tree: 4+2+1 = 7 cells
   - Coherent combining gain: 10·log(8) = 9 dB
   - Insertion loss: ~1.5 dB
   - Output: single combined signal at Ka-band

5. **RF AMP**
   - Post-combiner gain stage: Gain ≈ 12 dB, NF ≈ 4 dB (less critical after combiner)
   - Drives off-chip connection to next chip or downconverter

**Inter-chip combining:**
- RF_out from each chip connects to 32:1 off-chip combiner
- Total array gain: 10·log(256) = 24 dBi antenna gain + array directivity

**On-chip control (Ctr. Logic):**
- SPI interface to program phase codes per element
- MTP (Multi-Time Programmable Flash): stores calibration coefficients for 8 elements
- 6-bit DAC for I/Q weights, thermometer-coded for monotonicity

**Key specs (estimated from die size and process):**
- System NF: 3.5–4.5 dB total
- Scanning range: ±60° (grating lobe free with λ/2 element spacing)
- Data rate: supports 1–10 Gbps downlink (Ka-band DVB-S2X)
- Power per element: ~35 mW → 280 mW total per chip
- Radiation-hard design: ELT transistors, guard rings, TID > 30 krad"""
    },
    # --- MTP memory for calibration ---
    {
        "instruction": "What is MTP (Multi-Time Programmable) memory in RF CMOS chips and how is it used for phased array calibration in satellites?",
        "input": "",
        "output": """MTP (Multi-Time Programmable) Memory in RF Satellite Chips:

**What is MTP:**
MTP is a non-volatile analog memory technology implemented in standard CMOS processes without extra masks. Unlike OTP (one-time programmable) fuses, MTP can be erased and reprogrammed hundreds to thousands of times.

**Implementation in standard CMOS (e.g., GF180MCU):**
- Based on charge trapping in thin-oxide transistors (Fowler-Nordheim tunneling)
- Implemented using 5V/10V transistors available in mixed-signal processes
- Typical: FLOTOX (Floating Gate Oxide Transistor) cell — gate oxide 6–8 nm
- GF180MCU has native MTP IP via logic transistors; no extra process modules
- Alternative: standard EEPROM bitcell using two gate-poly floating gate structure

**Programming:**
- Write (erase+program): Apply VPP = 12–15V via on-chip charge pump to floating gate
- Read: sense floating gate charge via differential sense amplifier
- Endurance: 10,000–100,000 cycles (sufficient for in-orbit calibration)
- Data retention: 10–20 years at 85°C (critical for 5-year LEO mission lifetime)

**Use in phased array calibration:**
1. **Factory calibration** (ground): Measure each element's gain/phase error vs. frequency/temperature → compute correction codes → store in MTP
2. **In-orbit calibration**: Periodically re-calibrate using celestial sources (GPS or known satellite) → update MTP via telecommand uplink
3. **What is stored (per element, 6-8 elements per chip):**
   - Phase offset correction: 8-bit (1.4° LSB) × 8 elements = 64 bits
   - Gain correction: 6-bit × 8 elements = 48 bits
   - Temperature compensation coefficients: 16-bit × 8 = 128 bits
   - Total: ~240 bits per chip (~30 bytes) — very small MTP array

**Radiation hardness:**
- Floating gate cells use thick FOX oxide — less susceptible to TID than logic gates
- SEU on floating gate: discharge event possible from heavy ions (LET > 30 MeV·cm²/mg)
- Mitigation: ECC (Hamming code) on stored phase/gain words
- MTP write only done in safe (South Atlantic Anomaly-free) orbit windows"""
    },
]

print(f'Handcrafted Q&A entries: {len(HANDCRAFTED_QA)}')
for qa in HANDCRAFTED_QA:
    print(f'  - {qa["instruction"][:70]}...')

## 3. Convert AICircuit Dataset to Q&A Format

In [ ]:
AICIRCUIT_DIR = RAW_DIR / 'aicircuit'
aicircuit_qa = []

CIRCUIT_DESCRIPTIONS = {
    'lna': ('Low Noise Amplifier (LNA)', ['gain_db', 'nf_db', 'iip3_dbm', 'p1db_dbm', 'bw_ghz']),
    'pa':  ('Power Amplifier (PA)',       ['pout_dbm', 'pae_pct', 'gain_db', 'p1db_dbm']),
    'vco': ('Voltage Controlled Oscillator (VCO)', ['freq_ghz', 'phase_noise_dbc', 'kvco_mhz_v', 'pdiss_mw']),
    'mixer': ('Mixer',                   ['conversion_gain_db', 'nf_db', 'iip3_dbm', 'lo_leakage_dbm']),
    'rx':  ('Receiver chain',            ['gain_db', 'nf_db', 'iip3_dbm', 'bw_ghz']),
    'tx':  ('Transmitter chain',         ['pout_dbm', 'evm_pct', 'bw_ghz']),
}

if AICIRCUIT_DIR.exists():
    for csv_file in AICIRCUIT_DIR.rglob('*.csv'):
        circuit_key = csv_file.stem.lower()
        circuit_name, perf_cols = next(
            ((v) for k, v in CIRCUIT_DESCRIPTIONS.items() if k in circuit_key),
            ('RF Circuit', [])
        )
        try:
            df = pd.read_csv(csv_file)
            param_cols = [c for c in df.columns if c not in perf_cols]
            actual_perf = [c for c in perf_cols if c in df.columns]
            actual_param = [c for c in param_cols if c in df.columns]
            
            if not actual_param or not actual_perf:
                # Auto-detect: last N columns are performance metrics
                actual_param = list(df.columns[:len(df.columns)//2])
                actual_perf = list(df.columns[len(df.columns)//2:])
            
            for _, row in df.sample(min(200, len(df))).iterrows():
                param_str = ', '.join(f'{c}={row[c]:.4g}' for c in actual_param if c in row)
                perf_str = '\n'.join(f'- {c}: {row[c]:.4g}' for c in actual_perf if c in row)
                aicircuit_qa.append({
                    'instruction': f'Given these design parameters for a {circuit_name}, predict the circuit performance metrics.',
                    'input': f'Design parameters: {param_str}',
                    'output': f'Predicted performance:\n{perf_str}',
                    'source': f'aicircuit_{csv_file.stem}',
                })
        except Exception as e:
            print(f'  {csv_file.name}: {e}')

print(f'AICircuit Q&A pairs: {len(aicircuit_qa)}')

## 4. Convert HuggingFace EE Datasets

In [ ]:
hf_qa = []
HF_DIR = RAW_DIR / 'huggingface'

if HF_DIR.exists():
    for jsonl_file in HF_DIR.glob('*.jsonl'):
        with open(jsonl_file) as f:
            for line in f:
                try:
                    row = json.loads(line)
                    # STEM-AI EE format
                    if 'question' in row and 'answer' in row:
                        hf_qa.append({
                            'instruction': row['question'],
                            'input': row.get('context', ''),
                            'output': row['answer'],
                            'source': jsonl_file.stem,
                        })
                    # EEE-Bench format
                    elif 'problem' in row and 'solution' in row:
                        hf_qa.append({
                            'instruction': row['problem'],
                            'input': '',
                            'output': row['solution'],
                            'source': jsonl_file.stem,
                        })
                except Exception:
                    pass

print(f'HuggingFace Q&A pairs: {len(hf_qa)}')

## 5. Synthetic RF Q&A Generation via Local LLM (Ollama/Qwen3)

Generates additional RF design Q&A pairs using a locally-running Qwen3 model.
No cloud API required. If Ollama is not running, this section is skipped gracefully
and the pipeline continues with handcrafted + dataset pairs.

Start Ollama before running: `ollama serve && ollama pull qwen3:8b`

In [ ]:
import json, requests
from tqdm import tqdm

OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen3:8b"

SYSTEM_PROMPT = """You are an expert RF IC and antenna engineer with 20 years of experience 
designing Ka-band CMOS chips for satellite communications. Generate educational Q&A pairs 
that would help train an AI assistant to answer RF design questions accurately.
/no_think"""

RF_TOPICS = [
    "impedance matching networks at Ka-band using L-match, π-match, and transformer-based topologies",
    "noise figure analysis in cascaded RF systems using Friis formula",
    "CMOS LNA topologies: CS, CG, folded-cascode, noise-canceling — tradeoffs at mmWave",
    "phased array antenna beamforming: analog, digital, hybrid — for LEO satellite",
    "S-parameter measurement and interpretation: S11, S21, S12, S22 for RF circuits",
    "GDSFactory/GLayout Python code for RF passive components: inductors, MIM caps, transmission lines",
    "patch antenna design equations: width, length, feed, efficiency, bandwidth",
    "Ka-band satellite link budget: EIRP, FSPL, G/T, Eb/N0, rain fade margin",
    "VCO phase noise: Leeson's model, flicker noise upconversion, tank Q vs phase noise",
    "power amplifier linearity: IIP3, P1dB, ACPR, Doherty architecture for satellite TX",
    "radiation hardening of CMOS RF ICs: ELT transistors, guard rings, TID effects",
    "microstrip vs CPW vs stripline for Ka-band routing on PCB and chip",
    "CubeSat antenna constraints: size, mass, deployable arrays, Ka-band gain requirements",
    "silicon inductor design: Q factor, self-resonance frequency, substrate shielding techniques",
    "8:1 Wilkinson combiner tree design for phased array receivers at Ka-band",
    "switched-capacitor phase shifter design: bit weighting, NMOS switch sizing, insertion loss",
    "MTP non-volatile memory for satellite phased array calibration storage",
    "Ka-band satellite RX chain: LNA, buffer, phase shifter, combiner cascade NF",
]

def _ollama_available():
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=3)
        return r.status_code == 200
    except Exception:
        return False

def generate_rf_qa_ollama(topic: str, n_pairs: int = 5) -> list:
    prompt = f"""Generate {n_pairs} high-quality Q&A pairs about: {topic}

Format each pair as JSON on a single line:
{{"question": "...", "answer": "..."}}

Requirements:
- Questions should be specific and technical (not vague)
- Answers should include equations, numerical examples, design guidelines
- Focus on Ka-band (26-40 GHz) satellite IC context where applicable
- Include GLayout/gdsfactory code examples for layout-related topics

Output only the JSON objects, one per line, no other text."""

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n",
        "stream": False,
        "options": {"temperature": 0.7, "num_predict": 2048},
    }
    try:
        resp = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=120)
        resp.raise_for_status()
        text = resp.json().get("response", "")
    except Exception as e:
        print(f"  Ollama error: {e}")
        return []

    pairs = []
    for line in text.strip().split('\n'):
        line = line.strip()
        if line.startswith('{'):
            try:
                obj = json.loads(line)
                pairs.append({
                    'instruction': obj.get('question', ''),
                    'input': '',
                    'output': obj.get('answer', ''),
                    'source': 'synthetic_ollama',
                    'topic': topic,
                })
            except json.JSONDecodeError:
                pass
    return pairs

synthetic_qa = []
if _ollama_available():
    print(f"Ollama available at {OLLAMA_HOST} — generating synthetic RF Q&A with {OLLAMA_MODEL}")
    for topic in tqdm(RF_TOPICS, desc='Generating synthetic RF Q&A'):
        pairs = generate_rf_qa_ollama(topic, n_pairs=5)
        synthetic_qa.extend(pairs)
        print(f'  {topic[:55]}: {len(pairs)} pairs')
else:
    print(f"Ollama not running at {OLLAMA_HOST}. Start with: ollama serve && ollama pull {OLLAMA_MODEL}")
    print("Skipping synthetic generation — proceeding with handcrafted + dataset pairs only.")

synth_path = SYNTH_DIR / 'synthetic_rf_qa.jsonl'
with open(synth_path, 'w') as f:
    for qa in synthetic_qa:
        f.write(json.dumps(qa) + '\n')
print(f'Synthetic Q&A: {len(synthetic_qa)} pairs -> {synth_path}')

## 6. Merge All Sources into Final SFT Dataset

In [ ]:
def to_chatml(qa: dict) -> dict:
    """Convert to ChatML / Qwen chat format."""
    user_content = qa['instruction']
    if qa.get('input', '').strip():
        user_content += f"\n\nContext:\n{qa['input']}"
    return {
        'messages': [
            {'role': 'system', 'content': 'You are an expert RF IC and antenna engineer specializing in Ka-band CMOS chip design for satellite communications. Provide detailed, accurate technical answers with equations and design guidelines.'},
            {'role': 'user', 'content': user_content},
            {'role': 'assistant', 'content': qa['output']},
        ],
        'source': qa.get('source', 'unknown'),
    }

all_qa = []
all_qa.extend([{**qa, 'source': 'handcrafted'} for qa in HANDCRAFTED_QA])
all_qa.extend(aicircuit_qa)
all_qa.extend(hf_qa)

# Load synthetic if exists
synth_path = SYNTH_DIR / 'synthetic_rf_qa.jsonl'
if synth_path.exists():
    with open(synth_path) as f:
        all_qa.extend(json.loads(line) for line in f if line.strip())

# Filter: require non-empty instruction + output, min length
all_qa = [
    qa for qa in all_qa
    if len(qa.get('instruction', '')) > 20
    and len(qa.get('output', '')) > 50
]

random.shuffle(all_qa)
chatml_records = [to_chatml(qa) for qa in all_qa]

# Train/val split: 95/5
split = int(0.95 * len(chatml_records))
train_records = chatml_records[:split]
val_records = chatml_records[split:]

for fname, records in [('sft_train.jsonl', train_records), ('sft_val.jsonl', val_records)]:
    path = PROC_DIR / fname
    with open(path, 'w') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')
    print(f'{fname}: {len(records):,} records')

print(f'\nTotal SFT dataset: {len(chatml_records):,} examples')
print('Source breakdown:')
from collections import Counter
for src, cnt in Counter(r['source'] for r in chatml_records).most_common():
    print(f'  {src:30s}: {cnt:5d}')

## 7. PINN Dataset — Antenna Geometry → EM Performance

In [ ]:
import numpy as np

# Load Mendeley antenna dataset if available
ANTENNA_DIR = RAW_DIR / 'antenna_dataset'
pinn_df = None

for csv_file in ANTENNA_DIR.glob('*.xlsx'):
    pinn_df = pd.read_excel(csv_file)
    print(f'Loaded Mendeley antenna dataset: {pinn_df.shape}')
    print(pinn_df.describe())
    break

if pinn_df is None:
    print('Mendeley dataset not found — generating synthetic patch antenna data using analytical model')
    
    # Analytical patch antenna model (Pozar transmission line model)
    def patch_antenna_model(er, h_mm, W_mm, L_mm, f_ghz):
        c = 3e8
        f = f_ghz * 1e9
        h = h_mm * 1e-3
        W = W_mm * 1e-3
        L = L_mm * 1e-3
        
        # Effective permittivity
        eeff = (er+1)/2 + (er-1)/2 * (1 + 12*h/W)**-0.5
        
        # Resonant frequency
        DL = 0.412*h * (eeff+0.3)*(W/h+0.264) / ((eeff-0.258)*(W/h+0.8))
        f_res = c / (2*(L + 2*DL)*np.sqrt(eeff))
        
        # S11 at frequency f (simplified Lorentzian resonance)
        BW_frac = 3.77 * (er-1)/er**2 * h/L * W/L  # Pozar BW formula
        BW = f_res * BW_frac
        Q = f_res / BW
        s11_linear = abs((1j*(f-f_res)/f_res * 2*Q) / (1 + 1j*(f-f_res)/f_res * 2*Q))
        s11_db = 20*np.log10(max(s11_linear, 1e-6))
        
        # Gain estimate (includes substrate loss)
        rad_eff = 1 / (1 + 2*np.pi*f*h*np.sqrt(er)*0.0027/c)  # tanδ=0.0027
        gain_dbi = 10*np.log10(4*np.pi * W*L / (c/f)**2 * rad_eff)
        
        return f_res/1e9, s11_db, gain_dbi, rad_eff*100
    
    N = 10000
    rows = []
    for _ in range(N):
        er = np.random.uniform(2.2, 10.2)
        h  = np.random.uniform(0.2, 2.0)   # mm
        f  = np.random.uniform(5, 40)       # GHz
        lam_guided = (3e8 / (f*1e9)) / np.sqrt(er) * 1e3  # mm
        W  = np.random.uniform(0.6*lam_guided/2, 1.4*lam_guided/2)
        L  = np.random.uniform(0.4*lam_guided/2, 0.6*lam_guided/2)
        
        f_res, s11, gain, eff = patch_antenna_model(er, h, W, L, f)
        rows.append({'er': er, 'h_mm': h, 'W_mm': W, 'L_mm': L, 'f_ghz': f,
                     'f_res_ghz': f_res, 's11_db': s11, 'gain_dbi': gain, 'efficiency_pct': eff})
    
    pinn_df = pd.DataFrame(rows)
    print(f'Generated synthetic antenna dataset: {pinn_df.shape}')

pinn_df.to_csv(PROC_DIR / 'pinn_antenna.csv', index=False)
print(f'PINN dataset saved: {PROC_DIR / "pinn_antenna.csv"}')
pinn_df.describe()

## 8. Chipster + AnalogCoder → SFT Training Pairs

Load the pre-extracted PySpice SFT pairs from `01_pdf_extract.ipynb` Section 7  
and merge them into the main SFT corpus in ChatML format.

Sources:
- **Chipster** — your own PySpice analog generator code and notebooks  
- **AnalogCoder** — 24 analog circuit problems with expert PySpice solutions (AAAI 2025)


In [ ]:
# ── Load analog PySpice SFT pairs produced by 01_pdf_extract.ipynb §7 ───────
ANALOG_DIR = RAW_DIR / 'analog_pyspice'
analog_sft_path = ANALOG_DIR / 'sft_pairs.jsonl'

analog_qa = []
if analog_sft_path.exists():
    with open(analog_sft_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                analog_qa.append(row)
            except Exception:
                pass
    print(f'Loaded {len(analog_qa)} analog PySpice SFT pairs from {analog_sft_path.name}')
    # Source breakdown
    from collections import Counter
    counts = Counter(r.get('source','?') for r in analog_qa)
    for src, n in counts.items():
        print(f'  {src}: {n} pairs')
else:
    print(f'Not found: {analog_sft_path}')
    print('Run 01_pdf_extract.ipynb first (Section 7 clones + extracts both repos).')
    analog_qa = []

# ── Convert to ChatML format (same as all_qa list used in Section 6) ─────────
def analog_to_chatml(row: dict) -> dict:
    instruction = row.get('instruction', '')
    code_output  = row.get('output', '')
    circuit_name = row.get('circuit', 'analog circuit')
    source       = row.get('source', 'unknown')

    system_msg = (
        "You are an expert analog circuit designer specializing in PySpice simulation. "
        "When given a circuit specification, write complete, runnable Python code using "
        "PySpice that defines the netlist, configures ngspice simulation, and extracts "
        "key performance metrics (gain, bandwidth, noise figure, power consumption, etc.)."
    )

    return {
        "messages": [
            {"role": "system",    "content": system_msg},
            {"role": "user",      "content": instruction},
            {"role": "assistant", "content": code_output},
        ],
        "source":  source,
        "circuit": circuit_name,
    }

analog_chatml = [analog_to_chatml(r) for r in analog_qa if r.get('output','').strip()]
print(f'\nConverted {len(analog_chatml)} pairs to ChatML format')

# ── Append to main SFT JSONL ─────────────────────────────────────────────────
SFT_PATH = PROC_DIR / 'sft_rf_domain.jsonl'
existing = 0
if SFT_PATH.exists():
    with open(SFT_PATH) as f:
        existing = sum(1 for _ in f)

with open(SFT_PATH, 'a') as f:   # append — don't overwrite existing SFT data
    for item in analog_chatml:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

total = existing + len(analog_chatml)
print(f'SFT file: {existing} existing + {len(analog_chatml)} analog pairs = {total} total')
print(f'  -> {SFT_PATH}')

# ── Also add PySpice corpus to pre-training JSONL ────────────────────────────
PRETRAIN_PATH = PROC_DIR / 'pretrain_rf.jsonl'
corpus_path = ANALOG_DIR / 'corpus.py'
pretrain_added = 0

if corpus_path.exists():
    CHUNK_CHARS = 8192
    OVERLAP      = 512

    corpus_text = corpus_path.read_text(errors='ignore')
    start = 0
    chunks = []
    while start < len(corpus_text):
        chunk = corpus_text[start:start + CHUNK_CHARS].strip()
        if len(chunk) > 200:
            chunks.append(chunk)
        start += CHUNK_CHARS - OVERLAP

    with open(PRETRAIN_PATH, 'a') as f:
        for chunk in chunks:
            f.write(json.dumps({'text': chunk, 'source': 'analog_pyspice'}) + '\n')
    pretrain_added = len(chunks)
    print(f'\nPre-train corpus: +{pretrain_added} analog PySpice chunks -> {PRETRAIN_PATH.name}')

print('\nSample ChatML pair:')
if analog_chatml:
    ex = analog_chatml[0]
    print(f'  circuit: {ex["circuit"]} | source: {ex["source"]}')
    print(f'  user:    {ex["messages"][1]["content"][:120]}...')
    print(f'  assistant (first 200 chars): {ex["messages"][2]["content"][:200]}...')


In [ ]:
print('=== Final Dataset Summary ===')
for f in sorted(PROC_DIR.glob('*')):
    size = f.stat().st_size
    lines = sum(1 for _ in open(f, errors='ignore')) if f.suffix in ('.jsonl', '.csv') else 0
    print(f'  {f.name:35s}  {lines:6d} rows  {size/1e6:6.1f} MB')
print('\nNext: Run 03_qwen_sft_lora.ipynb to fine-tune Qwen')